In [ ]:
import pandas as pd
import numpy as np
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

### Quartz Dataset

In [5]:
quartz_df = pd.read_csv('../Data/QuaRTz/quartz_dataset.csv')
print("quartz dataset columns: ", quartz_df.columns)
quartz_df.head()

quartz dataset columns:  Index(['id', 'question', 'choices', 'answerKey', 'para', 'para_id',
       'para_anno', 'question_anno'],
      dtype='object')


,id,question,choices,answerKey,para,para_id,para_anno,question_anno
0,QRQA-10385-4,"John's town used to have lots of water, back w...","{'text': array(['scarce', 'plentiful'], dtype=...",A,Many of the worlds people live with water scar...,QRSent-10385,"{'effect_prop': 'population growth', 'cause_di...","{'more_effect_dir': 'several thousand', 'less_..."
1,QRQA-10116-3,Electrons further away from a nucleus have ___...,"{'text': array(['higher', 'lower'], dtype=obje...",A,"Electrons at lower energy levels, which are cl...",QRSent-10116,"{'effect_prop': 'energy', 'cause_dir_str': 'cl...","{'more_effect_dir': 'higher', 'less_effect_dir..."
2,QRQA-10208-3,Milo threw both a basketball and a baseball th...,"{'text': array(['basketball', 'baseball'], dty...",A,An object with greater mass or greater velocit...,QRSent-10208,"{'effect_prop': 'kinetic energy', 'cause_dir_s...","{'more_effect_dir': 'more', 'less_effect_dir':..."
3,QRQA-10012-3,If greenhouse gases in the atmosphere were to ...,"{'text': array(['cooler', 'warmer'], dtype=obj...",B,If there are more greenhouse gases in the atmo...,QRSent-10012,"{'effect_prop': 'Earth temperatures', 'cause_d...","{'more_effect_dir': 'warmer', 'less_effect_dir..."
4,QRQA-10136-4,If you put a lot of energy into some food the ...,"{'text': array(['increase', 'decrease'], dtype...",A,The temperature of matter increases with the a...,QRSent-10136,"{'effect_prop': 'temperature', 'cause_dir_str'...","{'more_effect_dir': 'increase', 'less_effect_d..."


In [31]:
print("id: ", quartz_df['id'][10])
print("question: ", quartz_df['question'][10])
print("choices: ", quartz_df['choices'][10])
print("answerKey: ", quartz_df['answerKey'][10])
print("para: ", quartz_df['para'][10])
print("para_id: ", quartz_df['para_id'][10])
print("para_anno: ", quartz_df['para_anno'][10])
print("question_anno: ", quartz_df['question_anno'][10])


id:  QRQA-10286-1-flip
question:  When there are fewer carbon atoms in something there are
choices:  {'text': array(['many options', 'few options'], dtype=object), 'label': array(['A', 'B'], dtype=object)}
answerKey:  B
para:  A: The more carbon atoms there are, the greater the number of possible arrangements of carbon atoms.
para_id:  QRSent-10286
para_anno:  {'effect_prop': 'options there are', 'cause_dir_str': 'more', 'effect_dir_str': 'greater', 'cause_dir_sign': 'MORE', 'effect_dir_sign': 'MORE', 'cause_prop': 'carbon atoms'}
question_anno:  {'more_effect_dir': 'many', 'less_effect_dir': 'few', 'less_cause_prop': 'carbon atoms', 'more_effect_prop': 'options', 'less_effect_prop': 'options', 'less_cause_dir': 'options'}


In [69]:
def normalize_choices(choices):
    data_dict = eval(choices, {"array": np.array, "np": np})
    structured_choice = {
        "choices": [t for t in data_dict['text']],
        "answers": [l for l in data_dict['label']]
    }
    return json.dumps(structured_choice, ensure_ascii=False)

In [93]:
quartz_dataset = pd.DataFrame(data={"question": quartz_df['question'], "choices": quartz_df['choices'].apply(normalize_choices), "answer": quartz_df['answerKey']})
quartz_dataset

,question,choices,answer
0,"John's town used to have lots of water, back w...","{""choices"": [""scarce"", ""plentiful""], ""answers""...",A
1,Electrons further away from a nucleus have ___...,"{""choices"": [""higher"", ""lower""], ""answers"": [""...",A
2,Milo threw both a basketball and a baseball th...,"{""choices"": [""basketball"", ""baseball""], ""answe...",A
3,If greenhouse gases in the atmosphere were to ...,"{""choices"": [""cooler"", ""warmer""], ""answers"": [...",B
4,If you put a lot of energy into some food the ...,"{""choices"": [""increase"", ""decrease""], ""answers...",A
...,...,...,...
2691,Yolanda moved to Denver where it is 5000 feet ...,"{""choices"": [""dense"", ""thin""], ""answers"": [""A""...",B
2692,The smallest alkenes should have the _____ boi...,"{""choices"": [""lowest"", ""highest""], ""answers"": ...",B
2693,If Milo is holding two objects close together ...,"{""choices"": [""increases"", ""decreases""], ""answe...",A
2694,If Jim is exploring layers of rock and he move...,"{""choices"": [""older"", ""younger""], ""answers"": [...",A
